# RukiAI Fine-tune on Colab T4

Runs the same `prepare_dataset.py` + `train.py` pipeline as the local setup, on a free Colab T4 (15 GB VRAM).

**Before running**: Runtime → Change runtime type → **T4 GPU**.

## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Upload the `fine-tuning/` folder

Two options — pick one:

**Option A (recommended): clone from GitHub** once you've pushed the project.

**Option B: zip the local folder** and upload it via Colab's file panel (skip the clone cell).

In [ ]:
# Option A — clone (edit the URL to your repo)
!git clone https://github.com/<YOUR_USER>/<YOUR_REPO>.git
%cd <YOUR_REPO>/fine-tuning

In [ ]:
# Option B — uploaded a zip via the file panel? Unzip it.
# !unzip -q fine-tuning.zip && %cd fine-tuning

## 3. Install Unsloth + deps

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 4. Build train.jsonl + val.jsonl

In [ ]:
!python3 prepare_dataset.py --strip-citations

## 5. Train

Defaults (in `train.py`) are sized for T4: `gemma-3-4b-it`, seq 2048, batch 2, grad-accum 4, 3 epochs. Roughly 30-60 min on a T4 for ~600 examples.

In [ ]:
!python3 train.py

## 6. Download GGUFs back to your laptop

After training finishes, the quantized models are in `output/gguf/`. Zip them and download.

In [ ]:
!ls -lh output/gguf/
!cd output && zip -r gguf.zip gguf/
from google.colab import files
files.download('output/gguf.zip')

## 7. Back on your laptop

```bash
cd /home/mors/Code/Ruki_Ai/fine-tuning
unzip -o gguf.zip -d output/
ollama create rukiai-gemma -f Modelfile
ollama run rukiai-gemma "What's the difference between PPF and NPS?"
```

Then in `backend/.env`: `ollama_model=rukiai-gemma`